# Cleaning the Animal Dataset
Cleans the raw messy CSV so we can chart animal **speed** by **diet**. Run every cell top to bottom; no manual edits needed.

In [ ]:
# Install word2number (converts English number words like "fifty-six" into 56).
# Colab doesn't include it by default; the ! runs this as a terminal command.
!pip install word2number

In [ ]:
import re                        # regular expressions: find patterns in text (e.g. "all the numbers in '24-30'")
import numpy as np               # numeric tools: np.mean (averages) and np.nan ("missing value")
import pandas as pd              # DataFrames: spreadsheet-like tables we can clean with code
from word2number import w2n      # w2n.word_to_num("fifty-six") -> 56

## 1. Upload the raw CSV
Pick the messy dataset file from your computer when prompted.

In [ ]:
from google.colab import files

# Opens a file picker in Colab; choose the raw messy CSV from your computer.
uploaded = files.upload()
# files.upload() returns {filename: file contents}; grab the name of the (first) uploaded file.
filename = list(uploaded.keys())[0]
# Read the CSV into a DataFrame (a table). We never edit the raw file by hand; all cleaning happens below in code.
raw = pd.read_csv(filename)
raw.head()  # preview the first 5 rows

## 2. Keep only the columns we need and give them simple names

In [ ]:
# Keep only the 3 columns this project needs (the raw file has 17 columns).
# .copy() makes a separate table, so changes below don't affect "raw".
df = raw[["Animal", "Average Speed (km/h)", "Diet"]].copy()
# Rename them in code (not by hand in the file) to the names the chart expects: name, speed, diet.
df.columns = ["name", "speed", "diet"]

## 3. Trim whitespace and drop rows with missing values

In [ ]:
for col in ["name", "speed", "diet"]:
    # Treat every value as text, then .str.strip() removes spaces at the start/end:
    # "           Carnivore           " -> "Carnivore"
    df[col] = df[col].astype("string").str.strip()

# Drop any row where name, speed, OR diet is missing (NaN), since we can't chart it.
df = df.dropna(subset=["name", "speed", "diet"])

## 4. Fix garbled characters in names
Some names were saved with the wrong text encoding (e.g. `GalÃ¡pagos` should be `Galápagos`).

In [ ]:
def fix_encoding(text):
    # "GalÃ¡pagos" happens when UTF-8 text is mistakenly read as Latin-1.
    # Undo it: turn the text back into the original bytes (latin-1), then read those bytes correctly (utf-8).
    try:
        return text.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text  # the name was already fine (normal names fail this conversion), so keep it as-is

# .apply runs fix_encoding on every name in the column.
df["name"] = df["name"].apply(fix_encoding)

## 5. Standardize diet
Some animals list several diets (e.g. `Carnivore, Scavenger`). We use the **first-listed (primary)** diet, lowercase it, and keep only carnivore / herbivore / omnivore.

In [ ]:
# "Carnivore, Scavenger" -> split on commas -> ["Carnivore", " Scavenger"] -> take the first -> "Carnivore"
# -> strip spaces -> lowercase -> "carnivore"
df["diet"] = df["diet"].str.split(",").str[0].str.strip().str.lower()
# Keep only rows whose diet is exactly one of the three we chart (drops insectivore, piscivore, scavenger, ...).
df = df[df["diet"].isin(["carnivore", "herbivore", "omnivore"])]

## 6. Clean speed
Handles values like `40`, `24-30` (range → average), `fifty-six` (words → number), and `16-24 (in water)` (drop the note). Anything else (e.g. `Varies`, `Not Applicable`) becomes NaN.

In [ ]:
def parse_speed(value):
    # 1. Remove notes in parentheses: "16-24 (in water)" -> "16-24"
    #    Regex \(.*?\) = an opening "(", any characters, then the nearest closing ")".
    value = re.sub(r"\(.*?\)", "", value).strip()

    # 2. Find every number in the text. Regex \d+(?:\.\d+)? = digits, optionally followed by a decimal part.
    #    "40" -> [40.0]    "24-30" -> [24.0, 30.0]    "0.02-0.03" -> [0.02, 0.03]
    numbers = [float(n) for n in re.findall(r"\d+(?:\.\d+)?", value)]
    if numbers:
        # One number stays the same; a range becomes its average: mean([24, 30]) = 27
        return np.mean(numbers)

    # 3. No digits found, so try reading it as English words: "fifty-six" -> 56
    try:
        return float(w2n.word_to_num(value))
    except ValueError:
        # 4. Not a speed at all (e.g. "Varies", "Not Applicable"): mark it as missing
        return np.nan

# Run parse_speed on every speed value, then drop rows whose speed couldn't be understood.
df["speed"] = df["speed"].apply(parse_speed)
df = df.dropna(subset=["speed"])

## 7. Remove duplicate animals (keep the first entry)

In [ ]:
# Some animals appear twice (e.g. Great White Shark: "fifty-six" and "56").
# Keep only the first row for each name. reset_index renumbers the rows 0, 1, 2, ... after all the removals.
df = df.drop_duplicates(subset="name", keep="first").reset_index(drop=True)

# Sanity checks: how many rows are left, how many of each diet, and the 10 fastest animals.
print(len(df), "clean rows")
print(df["diet"].value_counts())
df.sort_values("speed", ascending=False).head(10)

## 8. Export the cleaned CSV

In [ ]:
output_name = "Shwei - Cleaned Animal Data.csv"
# Save the cleaned table as a CSV. index=False stops pandas from adding an extra row-number column.
df.to_csv(output_name, index=False)
# Download it to your computer. Then copy it into the project as public/sample_animals.csv
# (files in public/ are served by the website, so the chart can load it from "/sample_animals.csv").
files.download(output_name)